In [1]:
# 2_generation_model_comparison.py

!pip install -q torch transformers sentence-transformers faiss-cpu pandas tqdm scikit-learn rouge-score nltk bert-score sacrebleu

import pandas as pd
import numpy as np
import torch
import faiss
import time
import nltk
import warnings
import logging
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from bert_score import score as bert_score
from sacrebleu.metrics import BLEU

# Reduce HuggingFace model init warnings
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

# NLTK downloads
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

warnings.filterwarnings("ignore")

# ------------------- DATA -------------------
df = pd.read_csv("/kaggle/input/mlops-amazon/amazon.csv")

documents = [
    f"""Product: {r['product_name']}
Price: {r['discounted_price']} | Rating: {r['rating']} ({r['rating_count']} reviews)
Description: {r['about_product']}"""
    for _, r in df.iterrows()
]

TEST_QUERIES = [
    {
        "query": "Recommend a good fast charging USB-C cable under 300 rupees",
        "reference": "The boAt A400 at ₹299 is recommended, supporting 24W-25W fast charging and having a slightly higher customer rating. For a tighter budget, the pTron Solero TB301 at ₹149 supports 15W fast charging.",
    },
    {
        "query": "Which cable has the highest rating and supports 60W charging?",
        "reference": "The Belkin USB-C to USB-C Fast Charging Cable (60W PD) is the best match, offering a strong 4.5-star rating along with full 60W fast-charging support.",
    },
    {
        "query": "What is the best iPhone lightning cable in the list?",
        "reference": "For super-fast charging with a USB-C adapter, the Belkin Lightning to USB-C is best. For a reliable and durable standard cable with a good warranty, the Duracell USB-A to Lightning is a great choice. The Hi-Mobiler is the most budget-friendly option.",
    },
    {
        "query": "Suggest me some good long lasting headphones",
        "reference": "The boAt Bassheads 100 in-ear wired earphones are recommended for longevity, featuring a premium coated wire for sturdiness and a 1-year warranty. They have a 4.1-star rating from over 360,000 reviews and cost between ₹349-₹379.",
    },
]


# ------------------- METRICS CLASS -------------------
class Metrics:
    def __init__(self):
        self.rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
        self.bleu = BLEU(effective_order=True)
        self.embedder = SentenceTransformer("all-MiniLM-L6-v2")

    def all(self, pred, ref, ctx):
        r = self.rouge.score(ref, pred)
        metrics = {
            "rouge_1_f1": r["rouge1"].fmeasure,
            "rouge_l_f1": r["rougeL"].fmeasure,
            "bleu": self.bleu.sentence_score(pred, [ref]).score / 100,
            "meteor": meteor_score([word_tokenize(ref.lower())], word_tokenize(pred.lower())),
        }
        P, R, F = bert_score(
            [pred], [ref], model_type="microsoft/deberta-large-mnli", verbose=False
        )
        metrics["bert_f1"] = F.mean().item()

        e1 = self.embedder.encode(pred)
        e2 = self.embedder.encode(ref)
        metrics["emb_sim"] = util.cos_sim(e1, e2).item()

        c_emb = self.embedder.encode(ctx)
        metrics["faith"] = min(1.0, 0.7 * util.cos_sim(e1, c_emb).item() + 0.3)

        return metrics

    def composite(self, m):
        w = {
            "rouge_1_f1": 0.1,
            "rouge_l_f1": 0.1,
            "bleu": 0.1,
            "meteor": 0.15,
            "bert_f1": 0.25,
            "emb_sim": 0.2,
            "faith": 0.1,
        }
        return sum(m[k] * w[k] for k in w)


metrics_calc = Metrics()


# ------------------- RAG CLASS -------------------
class RAG:
    def __init__(self, emb_name, gen_name):
        self.emb_name = emb_name
        self.gen_name = gen_name

        # Load embedding model
        print(f"Loading embedding model: {emb_name}")
        self.embedder = SentenceTransformer(emb_name)
        dim = self.embedder.encode(["test"]).shape[1]
        self.index = faiss.IndexFlatIP(dim)

        # Index documents
        print(f"Embedding {len(documents)} documents...")
        batches = [documents[i : i + 32] for i in range(0, len(documents), 32)]
        for b in tqdm(batches, desc="Indexing"):
            embs = self.embedder.encode(b, normalize_embeddings=True)
            self.index.add(embs)

        # Load generator model
        print(f"Loading generator: {gen_name}")
        torch.cuda.empty_cache()  # free memory before loading
        if "Qwen" in gen_name:
            tokenizer = AutoTokenizer.from_pretrained(gen_name)
            model = AutoModelForCausalLM.from_pretrained(
                gen_name,
                torch_dtype=torch.float16,
                device_map="auto",
                offload_folder="./offload",
            )
            self.generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
        else:
            self.generator = pipeline(
                "text-generation",
                model=gen_name,
                torch_dtype=torch.bfloat16 if "mistral" in gen_name else torch.float32,
                device_map="auto",
            )

    def retrieve(self, q, k):
        qe = self.embedder.encode([q], normalize_embeddings=True)
        D, I = self.index.search(qe, k)
        ctx = "\n\n".join([documents[i] for i in I[0]])
        return ctx

    def generate(self, q, ctx):
        prompt = f"Context:\n{ctx}\n\nQuestion: {q}\nAnswer:"
        out = self.generator(
            prompt,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.95,
            top_k=50,
            do_sample=True,
        )[0]["generated_text"]
        ans = out.split("Answer:")[-1].strip()
        return ans


# ------------------- EXPERIMENT -------------------
results = []
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"  # Fixed embedding
GENERATION_MODELS = [
    "Qwen/Qwen2.5-7B-Instruct",
    "google/flan-t5-large",
    "mistralai/Mistral-7B-Instruct-v0.2",
]

for gen in GENERATION_MODELS:
    print(f"\n{'='*80}\nTESTING GENERATOR: {gen}\n{'='*80}")
    try:
        torch.cuda.empty_cache()  # free memory before initializing RAG
        rag = RAG(EMBEDDING_MODEL, gen)
        for qd in TEST_QUERIES:
            ctx = rag.retrieve(qd["query"], k=5)
            ans = rag.generate(qd["query"], ctx)
            m = metrics_calc.all(ans, qd["reference"], ctx)
            m["composite"] = metrics_calc.composite(m)
            results.append({**m, "generation_model": gen, "query": qd["query"][:60]})
            print("\n------------------------------------------------------------")
            print(f"Generation Model: {gen}")
            print(f"Query: {qd['query']}")
            print("\nRetrieved Context (first 500 chars):")
            print(ctx[:500] + "...")
            print("\nGenerated Answer:")
            print(ans)
            print(f"\nComposite Score: {m['composite']:.4f}")
            print("------------------------------------------------------------\n")
        # Free memory after each generator
        del rag
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"Failed to load or run {gen}: {e}")

# Create results DataFrame and compute best generator
df_out = pd.DataFrame(results)
summary = df_out.groupby("generation_model")["composite"].mean().sort_values(ascending=False)

print("\n================ FINAL SUMMARY ================\n")
print("Average Composite Scores:")
print(summary)

best_model = summary.idxmax()
best_score = summary.max()
print("\n---------------------------------------------")
print(f"🏆 Best Generation Model: {best_model}")
print(f"🏅 Average Composite Score: {best_score:.4f}")
print("---------------------------------------------\n")

df_out.to_csv("2_generation_model_comparison.csv", index=False)
print("\nGeneration comparison saved → 2_generation_model_comparison.csv")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 83.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 72.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 24.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 67.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━

2025-12-05 07:12:40.418560: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764918760.612792      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764918760.658243      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

The following TP rules were not applied on any of the layers: {'layers.*.self_attn.q_proj': 'colwise', 'layers.*.self_attn.k_proj': 'colwise', 'layers.*.self_attn.v_proj': 'colwise', 'layers.*.self_attn.o_proj': 'rowwise', 'layers.*.mlp.gate_proj': 'colwise', 'layers.*.mlp.up_proj': 'colwise', 'layers.*.mlp.down_proj': 'rowwise'}
The following layers were not sharded: model.layers.*.post_attention_layernorm.weight, lm_head.weight, model.layers.*.self_attn.v_proj.weight, model.norm.weight, model.layers.*.input_layernorm.weight, model.layers.*.self_attn.q_proj.bias, model.layers.*.self_attn.k_proj.weight, model.embed_tokens.weight


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Device set to use cuda:0


Failed to load or run Qwen/Qwen2.5-7B-Instruct: CUDA out of memory. Tried to allocate 1.02 GiB. GPU 0 has a total capacity of 14.74 GiB of which 222.19 MiB is free. Process 4768 has 14.52 GiB memory in use. Of the allocated memory 13.35 GiB is allocated by PyTorch, and 1.04 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

================ FINAL SUMMARY ================

Average Composite Scores:
generation_model
mistralai/Mistral-7B-Instruct-v0.2    0.455265
google/flan-t5-large                  0.034202
Name: composite, dtype: float64

---------------------------------------------
🏆 Best Generation Model: mistralai/Mistral-7B-Instruct-v0.2
🏅 Average Composite Score: 0.4553
---------------------------------------------


Generation comparison saved → 